## DATA PIPELINE


In [1]:
import time
from pathlib import Path
import requests
import re
import csv
from collections import Counter
from __future__ import annotations
import pandas as pd
import numpy as np


#helper modules
from Python_code.modules import json_parser_theo as jp, classify_chats as cc, paser_chatti_html as hp

#pfade anlegen
repo = Path(".").resolve().parent
path_raw     = repo / "data/raw/data_sosci"
path_uploads = path_raw / "uploads"
csv_path     = path_raw / "daten.csv"
mapping_path = path_raw / "chatlog_mapping.csv"
uploads_root = path_raw / "uploads"
path_proc    = repo / "data/processed"

####################################
###Verwendung der Sosci APIs########
###################################
""" Um die daten abzurufen wird eine simple API verwednet, die nur für das data retrival einens Project funktioniert. Diese braucht für die Authentifizierung nur eine url (hier mit path_sosci_key gespeichert.
Um die hochgeladenen Datein herunter zu laden muss man die REST-API verwenden diese ist Personenspezifisch und braucht einen header (mit key) und url. Mt dieser API können datein einzeld herunter geladen werden und automatische in eine Ordnerstruktur eingebunden werden, die über ein Mapping festgehalten wird, für spätere verarbeitungensschritte"""

#pfade für keys
path_sosci_key        = repo / "API_KEYS/sosci_data_key.txt"
path_sosci_per_header = repo / "API_KEYS/sosci_per_header.txt"

#laden der url inclusive key
sosci_data_url = path_sosci_key.read_text().strip()

# Header-Wert robust einlesen
raw_header = path_sosci_per_header.read_text().strip()
if raw_header.lower().startswith("authorization:"):
    raw_header = raw_header.split(":", 1)[1].strip()

# Header definieren und urls
headers = {"Authorization": raw_header}
BASE = "https://survey.ifkw.lmu.de/admin/api.php?v1"
uploads_list_url = f"{BASE}/projects/5393/uploads" #wenn man ein / am Ende einfügt erzeugt das ein fehler

#CSV API-Abfrage
response = requests.get(sosci_data_url)
response.raise_for_status()    #Extrahiert den Chat-Verlauf und gibt ihn als csv zurück.
csv_path.write_bytes(response.content)
print(f"CSV gespeichert unter: {csv_path}")


#uploads-api-abfrage
r = requests.get(uploads_list_url, headers=headers)
r.raise_for_status()
filenames = r.json()["files"]
print(f"{len(filenames)} hochgeladene Dateien gefunden.")

#pattern um alle wichtigen infos aus html name zu ziehen
pattern = re.compile(r"^([A-Za-z]+\d+)\.(\d+)\.(\w+)$")
mapping_rows = []

for fname in filenames:
    match = pattern.match(fname)
    if not match:
        print(f"Dateiname passt nicht zum erwarteten Muster: {fname}")
        continue

    frage_code, teilnehmer_id, ext = match.groups()
    teilnehmer_id = str(int(teilnehmer_id))
    # Unterordner pro Person anlegen: uploads/<teilnehmer_id>/<fname>
    # wird später für den html_paser verwendent
    person_dir = path_uploads  / teilnehmer_id
    person_dir.mkdir(parents=True, exist_ok=True)
    out_path = person_dir / fname

    try:
        file_response = requests.get(f"{uploads_list_url}{"/"}{fname}", headers=headers)
        file_response.raise_for_status()
        out_path.write_bytes(file_response.content)
        print(f"Gespeichert: {out_path}")
        #mapping für weiterverarbeitung erweitern
        mapping_rows.append({"teilnehmer_id": teilnehmer_id, "frage_code": frage_code,
                              "dateiname": fname, "pfad": str(out_path)})
     #Fehler ausgeben statt abrechen, da wenn viel daten, einfach weitergearbeite wird
    except requests.exceptions.RequestException as e:
        print(f"Fehler bei {fname}: {e}")

    time.sleep(0.2) #sleep timer für den armen server

#mapping speicher, für weiterverarbeitung

with open(mapping_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["teilnehmer_id", "frage_code", "dateiname", "pfad"])
    writer.writeheader()
    writer.writerows(mapping_rows)

print(f"\nZuordnungstabelle gespeichert unter: {mapping_path}")

#Kontrolle wievele vhats per person
counts = Counter(row["teilnehmer_id"] for row in mapping_rows)
print("\nChats pro Teilnehmer-ID:")
for tid, n in sorted(counts.items()):
    print(f"  {tid}: {n} Datei(en)")

CSV gespeichert unter: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/daten.csv
10 hochgeladene Dateien gefunden.


Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/119/CH03.000119.html


Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/219/CH03.000219.html


Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/111/CH10.000111.html


Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/112/CH10.000112.html


...

Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/110/CH12.000110.html


Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/111/CH12.000111.html


Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/110/CH13.000110.html


Gespeichert: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/uploads/111/CH13.000111.html



Zuordnungstabelle gespeichert unter: /home/theo/PycharmProjects/Masterprojekt-Chatbots/data/raw/data_sosci/chatlog_mapping.csv

Chats pro Teilnehmer-ID:
  110: 3 Datei(en)
  111: 4 Datei(en)
  112: 1 Datei(en)
  119: 1 Datei(en)
  219: 1 Datei(en)


In [2]:

################
# Survey-CSV bereinigen (Zeit-/Metavariablen entfernen)
###############
df_raw = pd.read_csv(csv_path, sep="\t")

drop_exact = ["MAILSENT", "LASTDATA", "Q_VIEWER", "LASTPAGE", "MAXPAGE",
              "MODE", "STARTED", "REF", "QUESTNNR"]
time_cols  = [c for c in df_raw.columns if c.startswith("TIME")]   # TIME001..TIME_SUM
drop_cols  = [c for c in (drop_exact + time_cols) if c in df_raw.columns]

df_survey_clean = df_raw.drop(columns=drop_cols)
df_survey_clean.to_csv(path_proc / "survey_clean.csv", index=False)
print(f"Survey bereinigt: {df_raw.shape[1]} -> {df_survey_clean.shape[1]} Spalten "
      f"({len(drop_cols)} entfernt)")

###############################################################
# DATENSATZ 2: Chat-Nachrichten (JSON aus CSV + HTML-Uploads)#
################################################################

# Spalten mit JSON-Direkteingabe
JSON_INPUT_COLS = ["CH02s", "CH06s", "CH07s", "CH08s", "CH09s"]


# JSONs aus der CSV (Export-Prompt wird im Parser gefiltert) ----
rows_json = []
for _, r in df_raw.iterrows():
    tid = str(int(r["CASE"]))
    for col in JSON_INPUT_COLS:
        if col not in df_raw.columns or pd.isna(r[col]):
            continue
        messages = jp.extract_messages(str(r[col]))
        for turn, m in enumerate(messages, start=1):
            rows_json.append({
                "teilnehmer_id": tid,
                "frage_code":    col[:-1],          # CH02s -> CH02
                "quelle":        "json_csv", #zum überprüfen html oder json
                "turn":          turn,
                "role":          m["role"],
                "content":       m["content"],
            })
df_json = pd.DataFrame(rows_json)
#HTMLs aus den Uploads (via Mapping)
rows_html = []
mapping = pd.read_csv(mapping_path, dtype={"teilnehmer_id": str})
for _, r in mapping.iterrows():
    tid       = str(int(r["teilnehmer_id"]))
    html_path = uploads_root / tid / r["dateiname"]
    if not html_path.exists():
        print(f"HTML fehlt: {html_path}")
        continue
    for turn, m in enumerate(hp.extract(html_path), start=1):
        rows_html.append({
            "teilnehmer_id": tid,
            "frage_code":    r["frage_code"],
            "quelle":        "html_upload",
            "turn":          turn,
            "role":          m["speaker"],          # HTML-Parser: "speaker"/"text"
            "content":       m["text"],
        })
df_html = pd.DataFrame(rows_html)

# Zusammenführen + chat_id vergeben ----
df_chats = pd.concat([df_json, df_html], ignore_index=True)
df_chats["chat_uid"] = df_chats["teilnehmer_id"] + "_" + df_chats["frage_code"]
df_chats["chat_id"]  = df_chats.groupby("chat_uid").ngroup() + 1
df_chats = df_chats.sort_values(["chat_id", "turn"]).reset_index(drop=True)

df_chats.to_csv(path_proc / "chats_long.csv", index=False)
print(f"Chat-Nachrichten: {len(df_chats)} | eindeutige Chats: {df_chats['chat_id'].nunique()}")

Überspringe Turn 'conversation-turn-3' (nicht gerendert beim Export)


Überspringe Turn 'conversation-turn-4' (nicht gerendert beim Export)


Überspringe Turn 'conversation-turn-5' (nicht gerendert beim Export)


Survey bereinigt: 150 -> 100 Spalten (50 entfernt)


Überspringe Turn 'conversation-turn-3' (nicht gerendert beim Export)


...

Überspringe Turn 'conversation-turn-13' (nicht gerendert beim Export)


Überspringe Turn 'conversation-turn-14' (nicht gerendert beim Export)


Überspringe Turn 'conversation-turn-15' (nicht gerendert beim Export)


Überspringe Turn 'conversation-turn-3' (nicht gerendert beim Export)


Chat-Nachrichten: 96 | eindeutige Chats: 17


In [3]:
df_chats   = pd.read_csv(path_proc / "chats_long.csv")

df_labeled = cc.classify_chats(
    df_chats,
    model="gpt-5.5",
    api_key_path=repo / "API_KEYS/openai_key.txt",
)

df_labeled.to_csv(path_proc / "chats_labeled.csv", index=False)
df_labeled

    10/17 Chats klassifiziert ...


Fertig: 17 Chats klassifiziert.


,chat_id,teilnehmer_id,frage_code,task,sentiment,critical
0,1,110,CH02,Technische und analytische Unterstützung,Neutral,Nein
1,2,110,CH06,Informationssuche und Verständnis,Neutral,Nein
2,3,110,CH11,Schreiben und Textarbeit,Neutral,Nein
3,4,110,CH12,Technische und analytische Unterstützung,Neutral,Nein
4,5,110,CH13,Technische und analytische Unterstützung,Neutral,Nein
5,6,111,CH02,Informationssuche und Verständnis,Neutral,Nein
6,7,111,CH10,Technische und analytische Unterstützung,Freundlich,Nein
7,8,111,CH11,Technische und analytische Unterstützung,Freundlich,Ja
8,9,111,CH12,Praktische Unterstützung und Strukturierung,Freundlich,Nein
9,10,111,CH13,Technische und analytische Unterstützung,Neutral,Ja


In [4]:
# Mapping der Label-Strings auf die Zielspalten-Namen (Rohcounts)
TASK_TO_COL = {
    "Informationssuche und Verständnis":            "obs_info_n",
    "Schreiben und Textarbeit":                     "obs_schreiben_n",
    "Praktische Unterstützung und Strukturierung":  "obs_praktisch_n",
    "Technische und analytische Unterstützung":     "obs_technisch_n",
    "Lernen und Prüfungsvorbereitung":              "obs_lernen_n",
}
SENT_TO_COL = {
    "Freundlich":   "obs_sent_freundlich_n",
    "Neutral":      "obs_sent_neutral_n",
    "Unfreundlich": "obs_sent_unfreundlich_n",
}
CRIT_TO_COL = {
    "Ja":   "obs_kritisch_ja_n",
    "Nein": "obs_kritisch_nein_n",
}

ALL_COUNT_COLS = (list(TASK_TO_COL.values())
                  + list(SENT_TO_COL.values())
                  + list(CRIT_TO_COL.values()))


def aggregate_to_person(df_labeled: pd.DataFrame,
                        drop_unknown: bool = True,
                        verbose: bool = True) -> pd.DataFrame:
    df = df_labeled.copy()

    # Optionale Kontrolle/Meldung zu unknown-Labels vor der Aggregation
    if verbose:
        for col in ["task", "sentiment", "critical"]:
            n_unknown = (df[col] == "unknown").sum()
            if n_unknown:
                print(f"WARNUNG: {n_unknown} Chats mit unknown in '{col}'")

    if drop_unknown:
        before = len(df)
        df = df[(df["task"] != "unknown") &
                (df["sentiment"] != "unknown") &
                (df["critical"] != "unknown")].copy()
        if verbose and before != len(df):
            print(f"{before - len(df)} Chats mit unknown-Labels entfernt.")

    # Label-Strings in Zielspalten-Namen übersetzen
    df["task_col"] = df["task"].map(TASK_TO_COL)
    df["sent_col"] = df["sentiment"].map(SENT_TO_COL)
    df["crit_col"] = df["critical"].map(CRIT_TO_COL)

    persons = sorted(df["teilnehmer_id"].unique())
    result = pd.DataFrame(0, index=persons, columns=ALL_COUNT_COLS, dtype=int)
    result.index.name = "teilnehmer_id"

    # Rohcounts je Person hochzählen
    for _, r in df.iterrows():
        result.loc[r["teilnehmer_id"], r["task_col"]] += 1
        result.loc[r["teilnehmer_id"], r["sent_col"]] += 1
        result.loc[r["teilnehmer_id"], r["crit_col"]] += 1

    # Anzahl gültiger Chats pro Person
    result.insert(0, "n_chats_valid",
                  df.groupby("teilnehmer_id").size().reindex(persons).fillna(0).astype(int))

    result = result.reset_index()

    # Konsistenzchecks: die drei Kategoriengruppen müssen je zu n_chats_valid summieren
    task_sum = result[list(TASK_TO_COL.values())].sum(axis=1)
    sent_sum = result[list(SENT_TO_COL.values())].sum(axis=1)
    crit_sum = result[list(CRIT_TO_COL.values())].sum(axis=1)
    assert (task_sum == result["n_chats_valid"]).all(), "Task-Counts != n_chats_valid"
    assert (sent_sum == result["n_chats_valid"]).all(), "Sentiment-Counts != n_chats_valid"
    assert (crit_sum == result["n_chats_valid"]).all(), "Kritik-Counts != n_chats_valid"

    if verbose:
        print(f"Aggregiert: {len(result)} Personen.")
    return result

df_person = aggregate_to_person(df_labeled)

Aggregiert: 5 Personen.


In [5]:
df_person

,teilnehmer_id,n_chats_valid,obs_info_n,obs_schreiben_n,obs_praktisch_n,obs_technisch_n,obs_lernen_n,obs_sent_freundlich_n,obs_sent_neutral_n,obs_sent_unfreundlich_n,obs_kritisch_ja_n,obs_kritisch_nein_n
0,110,5,1,1,0,3,0,0,5,0,0,5
1,111,5,1,0,1,3,0,3,2,0,2,3
2,112,1,0,0,0,1,0,1,0,0,0,1
3,216,5,0,2,1,2,0,4,1,0,2,3
4,219,1,0,0,1,0,0,0,1,0,0,1


In [6]:

# 1:1-Umbenennungen
RENAME = {
    "CASE":               "id",
    "DE01":               "gender",
    "DE05":               "degree",
    "DE06":               "field",
    "sd_argument":        "sd_1",
    "sd_stressed":        "sd_2",
    "sd_listening":       "sd_3",
    "sd_advantage":       "sd_4",
    "sd_litter":          "sd_5",
    "sd_help":            "sd_6",
    "NU01":               "ai_experience",
    "NH01":               "freq",
    "IN01":               "info_literacy_where",
    "IN02":               "info_literacy_how",
    "info_use_info":      "info_use_1",
    "info_use_text":      "info_use_2",
    "info_use_structure": "info_use_3",
    "info_use_tech":      "info_use_4",
    "info_use_study":     "info_use_5",
    "IP01":               "inter_style",
    "KR01":               "crit_visible_chat",
    "SE01_01":            "self_assess_1",
    "SE01_02":            "self_assess_2",
    "SE01_03":            "self_assess_3",
}

# KI-Tools: SoSci T/F bzw. 2/1 -> 0/1
TOOL_MAP = {
    "NU02_02": "uses_gemini",
    "NU02_03": "uses_copilot",
    "NU02_05": "uses_deepseek",
    "NU02_06": "uses_claude",
}

MISSING_CODES = (-1, -9)


#SoSci-Mehrfachauswahl -> 0/1. Akzeptiert 'T'/'F' oder 2/1.
def _to_binary_tool(series: pd.Series) -> pd.Series:
    def conv(v):
        if isinstance(v, str):
            return 1 if v.strip().upper() == "T" else 0
        if pd.isna(v):
            return 0
        return 1 if int(v) == 2 else 0
    return series.map(conv)


def map_survey(df_raw: pd.DataFrame,
               recode_missing_to_na: bool = True) -> pd.DataFrame:
    df = df_raw.rename(columns={k: v for k, v in RENAME.items() if k in df_raw.columns}).copy()

    # age: Dropdown-Index -> echtes Alter (Index+15). Missing bleibt Missing.
    if "DE02" in df_raw.columns:
        idx = pd.to_numeric(df_raw["DE02"], errors="coerce")
        df["age"] = np.where(idx >= 3, idx + 15, np.nan)

    # KI-Tools binaer kodieren
    for src, tgt in TOOL_MAP.items():
        if src in df_raw.columns:
            df[tgt] = _to_binary_tool(df_raw[src])

    # Nur die relevanten Zielspalten behalten
    keep = ["id", "gender", "age", "degree", "field",
            "sd_1", "sd_2", "sd_3", "sd_4", "sd_5", "sd_6",
            "ai_experience", "uses_gemini", "uses_copilot", "uses_deepseek", "uses_claude",
            "freq", "info_literacy_where", "info_literacy_how",
            "info_use_1", "info_use_2", "info_use_3", "info_use_4", "info_use_5",
            "inter_style", "crit_visible_chat",
            "self_assess_1", "self_assess_2", "self_assess_3"]
    keep = [c for c in keep if c in df.columns]
    df = df[keep].copy()

    # SoSci-Missing-Codes (-1/-9) optional zu NA (age ausgenommen: -1 war schon behandelt)
    if recode_missing_to_na:
        for c in df.columns:
            if c in ("id", "age"):
                continue
            if pd.api.types.is_numeric_dtype(df[c]):
                df[c] = df[c].replace(list(MISSING_CODES), np.nan)

    df["id"] = df["id"].astype(int)
    return df


#mergerd suvery-Daten mit aggregierten Chatlog-Counts
def merge_survey_chatlogs(df_survey_mapped: pd.DataFrame,
                          df_person_counts: pd.DataFrame,
                          how: str = "inner") -> pd.DataFrame:
    counts = df_person_counts.copy()
    counts["id"] = counts["teilnehmer_id"].astype(int)
    counts = counts.drop(columns=["teilnehmer_id"])

    merged = df_survey_mapped.merge(counts, on="id", how=how)
    return merged

df_survey_mapped = map_survey(df_raw)
df_final         = merge_survey_chatlogs(df_survey_mapped, df_person, how="inner")

df_final.to_csv(path_proc / "perp_dataset.csv", index=False)
df_final

,id,gender,age,degree,field,sd_1,sd_2,sd_3,sd_4,sd_5,...,obs_info_n,obs_schreiben_n,obs_praktisch_n,obs_technisch_n,obs_lernen_n,obs_sent_freundlich_n,obs_sent_neutral_n,obs_sent_unfreundlich_n,obs_kritisch_ja_n,obs_kritisch_nein_n
0,110,1.0,20.0,2.0,3.0,1.0,1.0,1.0,4.0,4.0,...,1,1,0,3,0,0,5,0,0,5
1,111,2.0,24.0,2.0,3.0,3.0,2.0,4.0,5.0,5.0,...,1,0,1,3,0,3,2,0,2,3
2,112,2.0,24.0,2.0,3.0,3.0,2.0,4.0,5.0,5.0,...,0,0,0,1,0,1,0,0,0,1
3,216,1.0,30.0,2.0,5.0,4.0,4.0,4.0,2.0,1.0,...,0,2,1,2,0,4,1,0,2,3
4,219,2.0,29.0,2.0,7.0,2.0,2.0,4.0,4.0,4.0,...,0,0,1,0,0,0,1,0,0,1
